# Lesson 2.7 — experiments: heuristics for collecting parcels

Run the cells in order. Before each experiment, write down your prediction in the "Your prediction" cell (double-click it to edit), and only then run the code.

The first cell chooses the implementation:

- `"mine"` uses your `task.py`; solve Part B of the task first;
- `"reference"` uses the course's reference implementation. Set `IMPLEMENTATION = "reference"` if you want to run the notebook before finishing the task.

In `"reference"` mode "your" heuristic is the course's reference version, that is *h*<sub>3</sub>.

In [ ]:
%load_ext autoreload
%autoreload 2
import os

# "mine" uses your task.py; "reference" uses the course's reference implementation.
IMPLEMENTATION = os.environ.get("INTROAI_IMPLEMENTATION", "mine")
if IMPLEMENTATION == "mine":
    from task import parcel_heuristic
else:
    from introai.world.reference import parcel_heuristic

The function `heuristics(warehouse)` returns a dictionary of heuristics to compare: *h* = 0, the candidates *h*<sub>1</sub>–*h*<sub>4</sub> from the lesson, and yours. The function `solve` runs A\* with one of them and returns the plan, the number of expansions and the time in milliseconds. The parcel pickup problem comes from the course's reference version (lesson 1.3).

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt

from introai.search.reference import a_star
from introai.testing.search import true_costs
from introai.trace import TraceLimitExceeded, record
from introai.world.reference import CANDIDATE_HEURISTICS, ParcelPickupProblem
from introai.world.warehouse import WarehouseMap

MAPS = Path.cwd().parents[2] / "data" / "maps"
aisles = WarehouseMap.from_file(MAPS / "aisles.txt")
CELLS = [(3, 1), (5, 4), (9, 21), (13, 15), (1, 20), (7, 13)]


def yours(state):
    try:
        return parcel_heuristic(state)
    except NotImplementedError:
        raise RuntimeError(
            'parcel_heuristic is not written yet - set IMPLEMENTATION = "reference" in the '
            "first cell to run the experiments first"
        ) from None


def heuristics(warehouse):
    named = {"h = 0": lambda state: 0}
    for name in ("h1", "h2", "h3", "h4"):
        named[name] = CANDIDATE_HEURISTICS[name](warehouse)
    named["yours"] = yours
    return named


def solve(problem, h, limit=500_000):
    """A* with the heuristic h: (goal node, expansions, milliseconds)."""
    start = time.perf_counter()
    try:
        with record(limit=limit, kinds={"expand"}) as trace:
            node = a_star(problem, h)
    except TraceLimitExceeded:
        raise RuntimeError(f"A* expanded more than {limit} nodes") from None
    return node, len(trace.of_kind("expand")), (time.perf_counter() - start) * 1000

## Experiment 1: from 1 to 6 parcels

A\* collects the first 1, 2, …, 6 parcels of the list `CELLS` on the map with aisles, each time with a different heuristic.

**Your prediction.** Order the heuristics from the one that expands the most nodes to the one that expands the fewest. Which of them will give a plan that is too long?

*My prediction:* …

In [ ]:
counts = list(range(1, len(CELLS) + 1))
work = {}
print(f"{'parcels':>7} " + " ".join(f"{name:>12}" for name in heuristics(aisles)))
for k in counts:
    warehouse = aisles.with_parcels(CELLS[:k])
    cells = []
    best = None
    for name, h in heuristics(warehouse).items():
        node, expanded, _ = solve(ParcelPickupProblem(warehouse), h)
        work.setdefault(name, []).append(expanded)
        best = node.path_cost if best is None else min(best, node.path_cost)
        cells.append((expanded, node.path_cost))
    row = [f"{e}/{c:g}" + ("!" if c > best else "") for e, c in cells]
    print(f"{k:>7} " + " ".join(f"{text:>12}" for text in row))
print("expansions/plan length; ! marks a plan longer than the shortest one")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
series = [
    ("h = 0", "#2a78d6", "o"),
    ("h1", "#eb6834", "s"),
    ("h2", "#1baf7a", "^"),
    ("h3", "#eda100", "D"),
    ("yours", "#e87ba4", "v"),
]
for name, color, marker in series:
    ax.plot(
        counts,
        work[name],
        color=color,
        linewidth=2,
        marker=marker,
        markersize=8,
        markeredgecolor="#fcfcfb",
        markeredgewidth=1.5,
        label=name,
    )
# Direct labels at the line ends, nudged apart on the log scale so they never overlap.
label_y = None
for name, *_ in sorted(series, key=lambda item: work[item[0]][-1]):
    label_y = work[name][-1] if label_y is None else max(work[name][-1], label_y * 1.3)
    ax.annotate(
        name,
        xy=(counts[-1], work[name][-1]),
        xytext=(counts[-1] + 0.15, label_y),
        textcoords="data",
        va="center",
        color="#0b0b0b",
    )
ax.set_yscale("log")
ax.set_title("Nodes expanded by A* (admissible heuristics only)", loc="left", color="#0b0b0b")
ax.set_xlabel("number of parcels", color="#52514e")
ax.set_ylabel("nodes expanded (log scale)", color="#52514e")
ax.margins(x=0.12)
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color("#c3c2b7")
ax.tick_params(colors="#52514e")
ax.legend(frameon=False, loc="upper left", labelcolor="#0b0b0b")
fig.tight_layout()

**Think about it.** Why does *h*<sub>4</sub> expand the fewest nodes and still is useless if we want the shortest plan? How much better is *h*<sub>3</sub> than *h*<sub>2</sub> with six parcels? Where is your heuristic on the chart?

## Experiment 2: dominance

We compare the heuristics state by state: in all 1240 states reachable with three parcels.

**Your prediction.** Is *h*<sub>3</sub> greater than or equal to *h*<sub>2</sub> in every state? How does your heuristic compare with *h*<sub>3</sub>?

*My prediction:* …

In [ ]:
warehouse = aisles.with_parcels(CELLS[:3])
states = list(true_costs(ParcelPickupProblem(warehouse)))
named = heuristics(warehouse)


def compare(first, second):
    larger = sum(named[first](s) > named[second](s) for s in states)
    smaller = sum(named[first](s) < named[second](s) for s in states)
    print(
        f"{first} vs {second}: larger in {larger}, smaller in {smaller}, "
        f"equal in {len(states) - larger - smaller} of {len(states)} states"
    )


compare("h3", "h2")
compare("h2", "h1")
compare("yours", "h3")

**Think about it.** What does the result "smaller in 0" mean? Why is *h*<sub>3</sub> never smaller than *h*<sub>2</sub>? If your heuristic is larger than *h*<sub>3</sub> in some states and smaller in others, which heuristic is worth building from both?

## Experiment 3: expansions and time

With six parcels we measure how many milliseconds A\* takes with each heuristic (the median of three runs).

**Your prediction.** Will the order by time be the same as the order by the number of expansions?

*My prediction:* …

In [ ]:
warehouse = aisles.with_parcels(CELLS)
for name, h in heuristics(warehouse).items():
    runs = sorted((solve(ParcelPickupProblem(warehouse), h) for _ in range(3)), key=lambda r: r[2])
    node, expanded, ms = runs[1]
    print(f"{name:>6}: {expanded:>6} expansions, {ms:>7.1f} ms, plan {node.path_cost:g} actions")

**Think about it.** Does any heuristic expand fewer nodes and still not run faster? How much does computing *h*<sub>3</sub> for one state cost compared with *h*<sub>1</sub>? Times depend on the computer; compare them with each other, not with your classmates' numbers.

## Summary

Go back to the review questions in the lesson. After your own attempt, you can compare your results and conclusions with the answers in `solutions/02_search/07_designing_heuristics/answers.md`.